In [ ]:
#!/usr/bin/env python3
"""
Summarize TCGA-BRCA survival metrics.

Analogue of tea_seq/orch-regression/notebooks/create_summary_split_3.ipynb,
but for survival tasks. Shows train/test C-index where available.
"""

from __future__ import annotations
import json
from pathlib import Path
import numpy as np
import pandas as pd

# ---- Settings ----
split_tag = "brca_survival"
results_root = Path("../results/brca_survival")
out_dir = results_root / "summary"
out_dir.mkdir(parents=True, exist_ok=True)

csv_path = out_dir / f"{split_tag}_combined_survival_metrics.csv"
tex_path = out_dir / f"{split_tag}_combined_survival_metrics.tex"


def load_json(path: Path) -> dict | None:
    if not path.exists():
        return None
    with open(path) as f:
        return json.load(f)


def first_present(*vals):
    for v in vals:
        if v is not None:
            return v
    return None


def latest_intermediate_fusion_spec(results_root: Path) -> dict | None:
    """
    orchamp_brca_survival_intermediate_fusion.ipynb writes results to
    intermediate_fusion_k{n_clusters}, where n_clusters comes from the gap
    statistic and can differ between runs. Auto-detect every
    intermediate_fusion_k* folder present and use whichever metrics.json has
    the most recent "timestamp", so this summary always reflects the latest
    run without needing a hardcoded folder name.
    """
    candidates = []
    for folder in sorted(results_root.glob("intermediate_fusion_k*")):
        d = load_json(folder / "metrics.json")
        if d is None or "timestamp" not in d:
            continue
        candidates.append((d["timestamp"], folder, d))
    if not candidates:
        return None
    candidates.sort(key=lambda t: t[0])
    _, latest_folder, latest_metrics = candidates[-1]
    if len(candidates) > 1:
        skipped = ", ".join(f.name for _, f, _ in candidates[:-1])
        print(f"[INFO] Multiple intermediate_fusion_k* runs found; using latest: "
              f"{latest_folder.name} (skipped: {skipped})")
    return {
        "folder": latest_folder.name,
        "label": f"DAIF-CKA (k={latest_metrics.get('n_clusters', '?')})",
        "metrics": "metrics.json",
    }


# Ordered as desired in the output table.
# For methods with train/test split across folders, provide train_metrics separately.
METHODS = [
    {"folder": "early_fusion", "label": "OrchAMP (Early Fusion)", "metrics": "metrics.json"},
]

_intermediate_spec = latest_intermediate_fusion_spec(results_root)
if _intermediate_spec is not None:
    METHODS.append(_intermediate_spec)
else:
    print("[SKIP] No intermediate_fusion_k* results found under", results_root)

METHODS += [
    {"folder": "late_fusion", "label": "EB-PCA (Late Fusion)", "metrics": "metrics.json"},
    {"folder": "mofa", "label": "MOFA+", "metrics": "metrics.json"},
    {"folder": "multigrate_test", "label": "Multigrate", "metrics": "metrics.json",
     "train_folder": "multigrate_train", "train_metrics": "metrics_train.json"},
]


records = []
for spec in METHODS:
    folder = results_root / spec["folder"]
    d = load_json(folder / spec.get("metrics", "metrics.json"))
    if d is None:
        print(f"[SKIP] No metrics JSON found: {folder}")
        continue

    train_d = None
    if "train_folder" in spec:
        train_d = load_json(results_root / spec["train_folder"] / spec.get("train_metrics", "metrics_train.json"))

    c_test = d.get("c_index_test")
    c_train = first_present(d.get("c_index_train"), None if train_d is None else train_d.get("c_index_train"))
    n_train = first_present(d.get("n_train"), None if train_d is None else train_d.get("n_train"))
    n_test = d.get("n_test")
    n_events_train = first_present(d.get("n_events_train"), None if train_d is None else train_d.get("n_events_train"))
    n_events_test = d.get("n_events_test")

    rec = {
        "Method": spec["label"],
        "Train C-index": c_train,
        "Test C-index": c_test,
        "n_train": n_train,
        "events_train": n_events_train,
        "n_test": n_test,
        "events_test": n_events_test,
        "folder": spec["folder"],
    }
    records.append(rec)

    tr = f"{c_train:.4f}" if c_train is not None else "N/A"
    te = f"{c_test:.4f}" if c_test is not None else "N/A"
    print(f"[OK] {spec['label']:22s}  Train C={tr}  Test C={te}")

df = pd.DataFrame.from_records(records).set_index("Method")
df.to_csv(csv_path)
print(f"\n[Saved] CSV -> {csv_path}")

print("\n--- Summary Table ---")
print(df[["Train C-index", "Test C-index", "n_train", "events_train", "n_test", "events_test"]].to_string(float_format="{:.4f}".format))


In [ ]:
# ------------------------------------------------------------
# LaTeX table
# ------------------------------------------------------------

cols = ["Train C-index", "Test C-index"]

def fmt_num(v, bold=False):
    if v is None or pd.isna(v):
        return "--"
    s = f"{float(v):.4f}"
    return f"\\textbf{{{s}}}" if bold else s

best_test = df["Test C-index"].dropna().max() if not df.empty else np.nan

lines = [
    "% Auto-generated: TCGA-BRCA survival benchmark summary",
    "% Requires: \\usepackage{booktabs}",
    "\\begin{table}[!htbp]",
    "\\centering",
    "\\small",
    "\\begin{tabular}{lrrrr}",
    "\\toprule",
    "Method & Train C-index & Test C-index & Test events & Test $n$ " + (chr(92) * 2),
    "\\midrule",
]

for method in df.index:
    c_train = df.at[method, "Train C-index"]
    c_test = df.at[method, "Test C-index"]
    test_events = df.at[method, "events_test"]
    n_test = df.at[method, "n_test"]
    bold_test = pd.notna(c_test) and pd.notna(best_test) and abs(float(c_test) - float(best_test)) < 1e-12
    row = [
        method,
        fmt_num(c_train),
        fmt_num(c_test, bold=bold_test),
        "--" if pd.isna(test_events) else str(int(test_events)),
        "--" if pd.isna(n_test) else str(int(n_test)),
    ]
    lines.append(" & ".join(row) + " " + (chr(92) * 2))

lines += [
    "\\bottomrule",
    "\\end{tabular}",
    "\\caption{TCGA-BRCA overall survival prediction. Higher C-index is better; bold indicates the best test C-index.}",
    "\\label{tab:tcga_brca_survival}",
    "\\end{table}",
]

latex = "\n".join(lines)
with open(tex_path, "w") as f:
    f.write(latex)
print(f"[Saved] LaTeX -> {tex_path}")
print(latex)


In [ ]:
# ------------------------------------------------------------
# Optional: compact ranking by test C-index
# ------------------------------------------------------------

rank_path = out_dir / f"{split_tag}_survival_test_cindex_ranking.csv"
rank_df = df.sort_values("Test C-index", ascending=False)
rank_df[["Test C-index", "Train C-index", "n_test", "events_test", "folder"]].to_csv(rank_path)
print(f"[Saved] Ranking -> {rank_path}")
print(rank_df[["Test C-index", "Train C-index"]].to_string(float_format="{:.4f}".format))
